In [44]:
import pandas as pd
import numpy as np
from scipy.linalg import eigh
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment

In [45]:
trainIDs =  ["F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100epat2_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",
            "F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100epat2_seed1993_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",
            "F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100epat2_seed42_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",
            "F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100ep_seed1994_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",
            "F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100ep_seed2023_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",]

merge_cannonical_pca = False #Apply PCA to merge the cannonical components, if False, then just average them.

res_root = r"/project/ukbblatent/Out/Results"
gwas_folder = "GWAS_fullDSV2"

In [46]:
def align_latents(models):
    aligned_dfs = []
    base_df = models[0]
    
    for df in models[1:]:
        aligned_columns = []
        
        # Calculate the similarity matrix
        similarity_matrix = abs(cosine_similarity(base_df.T, df.T))
        
        # Use the Hungarian Algorithm to find the optimal alignment
        row_ind, col_ind = linear_sum_assignment(-similarity_matrix)
        
        for base_col, aligned_col in zip(row_ind, col_ind):
            aligned_column = df.iloc[:, aligned_col]
            aligned_column.name = base_df.columns[base_col]
            aligned_columns.append(aligned_column)
        
        aligned_df = pd.concat(aligned_columns, axis=1)
        aligned_dfs.append(aligned_df)
        
    return [base_df] + aligned_dfs

In [47]:
def mcca(models, n_latents):
    n_models = len(models)
    
    for i in range(n_models):
        if type(models[i]) is pd.DataFrame:
            models[i] = models[i].to_numpy()
    
    print("Calculating cross-covariance matrices between all pairs of models...")
    cross_covs = []
    for i in range(n_models):
        for j in range(i, n_models):
            cov = np.dot(models[i].T, models[j]) / (len(models[i]) - 1)
            cross_covs.append(cov)
    
    print("Creating block diagonal matrix....")
    D = np.zeros((n_latents * n_models, n_latents * n_models))
    for i, cov_matrix in enumerate(cross_covs):
        row = i // n_models
        col = i % n_models
        D[row*n_latents:(row+1)*n_latents, col*n_latents:(col+1)*n_latents] = cov_matrix
        if row != col:
            D[col*n_latents:(col+1)*n_latents, row*n_latents:(row+1)*n_latents] = cov_matrix.T

    print("Creating identity matrices block (I ⊗ I) where I is the identity matrix of size (n_latents x n_latents)...")
    I = np.identity(n_latents)
    I_block = np.kron(np.eye(n_models), I)

    print("Solving generalised eigenvalue problem...")
    eigenvalues, eigenvectors = eigh(D, I_block)
    
    print("Sorting eigenvectors based on eigenvalues....")
    sorted_indices = np.argsort(-eigenvalues)  
    sorted_eigenvectors = eigenvectors[:, sorted_indices]
    top_eigenvectors = sorted_eigenvectors[:, :n_latents]
    
    print("Extracting canonical components for each model....")
    canonical_components = []
    for i in range(n_models):
        start = i * n_latents
        end = (i + 1) * n_latents
        weights = top_eigenvectors[start:end, :]
        components = np.dot(models[i], weights)
        canonical_components.append(components)

    return canonical_components

In [50]:
print("Reading the latents.....")
processed_latents = []
for trainID in trainIDs:
    df = pd.read_table(f"{res_root}/{trainID}/{gwas_folder}/processed_raw.tsv").sort_values("FID").reset_index(drop=True)
    IID = df.IID
    FID = df.FID
    df.drop(columns=["IID", "FID"], inplace=True)
    latents = df.columns.to_list()
    processed_latents.append(df)
    
print("Aligning the latents....")
processed_latents = align_latents(processed_latents)

print("Bringing the latents to the canonical components....")
canonical_components = mcca(processed_latents, n_latents=len(latents))

print("Merging the cannonical components....")
if merge_cannonical_pca:
    print("....with PCA")
    concatenated_canonical_components = np.concatenate(canonical_components, axis=1)
    pca = PCA(n_components=len(latents))
    final_components = pca.fit_transform(concatenated_canonical_components)
else:
    print("....with average")
    stacked_canonical_components = np.stack(canonical_components, axis=2)
    final_components = np.mean(stacked_canonical_components, axis=2)

merged_latents = pd.DataFrame(final_components, columns=latents)
merged_latents['IID'] = IID
merged_latents['FID'] = FID

cols = ['FID', 'IID'] + [col for col in merged_latents.columns if col not in ['FID', 'IID']]
merged_latents = merged_latents[cols]


Reading the latents.....


Aligning the latents....
Bringing the latents to the canonical components....
Calculating cross-covariance matrices between all pairs of models...
Creating block diagonal matrix....
Creating identity matrices block (I ⊗ I) where I is the identity matrix of size (n_latents x n_latents)...
Solving generalised eigenvalue problem...
Sorting eigenvectors based on eigenvalues....
Extracting canonical components for each model....
Merging the cannonical components....
....with average


In [49]:
if merge_cannonical_pca:
    merged_latents.to_csv("/group/glastonbury/soumick/GWAS/merged_models/5Seeds_FVAE_128_Crop3D_DSV2_fold0/align_mcca_pca/merged_latents_raw.tsv", sep="\t", index=False)
else:
    merged_latents.to_csv("/group/glastonbury/soumick/GWAS/merged_models/5Seeds_FVAE_128_Crop3D_DSV2_fold0/align_mcca_avg/merged_latents_raw.tsv", sep="\t", index=False)

#Mathematical defination

0. (Align the latents) 

\text{Maximise} \quad S = \sum_{i=1}^{n} \cos(\theta_{i, \pi(i)})

\text{Where } \cos(\theta_{i, \pi(i)}) \text{ is the cosine similarity between the } i^{th} \text{ column of } \mathbf{X} \text{ and the } \pi(i)^{th} \text{ column of } \mathbf{Y}.

\text{The permutation } \pi \text{ is found such that } S \text{ is maximised.}


1. The cross-covariance between every pair of datasets \(X_i\) and \(X_j\) is calculated as:
\[
\text{cov}_{ij} = \frac{X_i^T X_j}{n-1}
\]

2. A block-diagonal matrix \(D\) is constructed such that each block \(D_{ij}\) is either a covariance matrix \(\text{cov}_{ij}\) for \(i = j\), or a cross-covariance matrix for \(i \neq j\):
\[
D = \begin{pmatrix}
\text{cov}_{11} & \text{cov}_{12} & \cdots & \text{cov}_{1m} \\
\text{cov}_{21} & \text{cov}_{22} & \cdots & \text{cov}_{2m} \\
\vdots  & \vdots  & \ddots & \vdots  \\
\text{cov}_{m1} & \text{cov}_{m2} & \cdots & \text{cov}_{mm} 
\end{pmatrix}
\]

3. An identity block matrix \(I \otimes I\) is then created, where \(\otimes\) represents the Kronecker product.
\[
I \otimes I = I_{\text{n\_models}} \otimes I_{\text{n\_latents}}
\]

4. The generalised eigenvalue problem is solved as follows:
\[
Dv = \lambda (I \otimes I) v
\]

5. Eigenvectors are sorted based on their eigenvalues in descending order:
\[
\lambda_{\text{sorted}} = \text{sort}(\lambda)
\]
Only the top \(n_{\text{latents}}\) are selected.

6. The canonical components \(C_i\) for each dataset \(X_i\) are calculated as:
\[
C_i = X_i W_i
\]
Where \(W_i\) are the selected weights (eigenvectors) corresponding to dataset \(i\).

7. Principal Component Analysis (PCA) is applied to the concatenated canonical components \(C = [C_1, C_2, \ldots, C_m]\).

These equations provide a mathematical explanation of the code's steps, excluding data conversion.